# C++ CPU × CUDA GPU — real hardware

1. Open this notebook in Colab.
2. Select an **NVIDIA GPU** runtime, if available.
3. Run the cells in order.
4. Upload `dist/cpu_gpu_cuda.zip` when prompted.

GPU availability varies. The CPU being measured belongs to the same remote machine as the GPU. The notebook contains no fabricated CUDA results.

In [ ]:
import subprocess

# Stop if the runtime cannot see an NVIDIA driver/GPU.
subprocess.run(["nvidia-smi"], check=True)
subprocess.run(["nvcc", "--version"], check=True)


In [ ]:
from google.colab import files
from pathlib import Path
import io
import zipfile
import sys

# Upload only the package created by python/prepare_demo.py.
uploaded = files.upload()
payload = uploaded["cpu_gpu_cuda.zip"]
base = Path("/content").resolve()

with zipfile.ZipFile(io.BytesIO(payload)) as archive:
    # Reject paths that escape the extraction directory.
    for member in archive.infolist():
        target = (base / member.filename).resolve()
        if not target.is_relative_to(base):
            raise ValueError("Invalid path in ZIP archive.")
    archive.extractall(base)

ROOT = base / "cpu_gpu_cuda"
sys.path.insert(0, str(ROOT / "python"))


## Compile C++ and CUDA together
`native/gpu.cu` contains naive and tiled matrix products with shared buffers and barriers. The optimized CPU uses OpenMP in this Linux build. The executable fails if numerical validation fails.

In [ ]:
import os
from build_native import build

os.environ["OMP_NUM_THREADS"] = "4"
binary = build(cpu_only=False)
subprocess.run([str(binary), "--self-test", "--require-gpu"], check=True)


## Measure the same model on both architectures
Weights and buffers are prepared before timing. This run requires an actual GPU. It uses five warmups and 21 repetitions per batch.

In [ ]:
from analysis import run_benchmark, analyze
from IPython.display import Image, display, Markdown
%matplotlib inline

path = run_benchmark(binary, ROOT / "results" / "native", batches=[1, 8, 64, 128, 256, 1024], repetitions=21, warmup=5, require_gpu=True)
rows = analyze(path)
display(Image(filename=str(path.parent / "comparison_real.png")))
display(Markdown((path.parent / "analysis.md").read_text(encoding="utf-8")))


## Download results
The ZIP includes CSV, report, chart, GPU description, and host metadata. Keep percentiles and configuration when presenting results. Tiled CUDA can lose to the naive kernel on small matrices; the code does not force a winner.

In [ ]:
import shutil

archive = shutil.make_archive("/content/cuda_results", "zip", path.parent)
files.download(archive)
